In [61]:
import gym
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical

In [62]:
## Initilaize Data store, Actor Network and Critic network
############################# Data Store ####################################################
class PPOMemory():
    def  __init__(self, batch_size):
        self.states = []
        self.actions= []
        self.action_probs = []
        self.rewards = []
        self.vals = []
        self.dones = []

        self.batch_size = batch_size

    def generate_batches(self):
        ## suppose n_states=20 and batch_size = 4
        n_states = len(self.states)
        ##n_states should be always greater than batch_size
        ## batch_start is the starting index of every batch
        ## eg:   array([ 0,  4,  8, 12, 16]))
        batch_start = np.arange(0, n_states, self.batch_size)
        ## random shuffling if indexes
        # eg: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]
        indices = np.arange(n_states, dtype=np.int64)
        ## eg: array([12, 17,  6,  7, 10, 11, 15, 13, 18,  9,  8,  4,  3,  0,  2,  5, 14,19,  1, 16])
        np.random.shuffle(indices)
        batches = [indices[i:i+self.batch_size] for i in batch_start]
        ## eg: [array([12, 17,  6,  7]),array([10, 11, 15, 13]),array([18,  9,  8,  4]),array([3, 0, 2, 5]),array([14, 19,  1, 16])]
        return np.array(self.states),np.array(self.actions),\
               np.array(self.action_probs),np.array(self.vals),np.array(self.rewards),\
               np.array(self.dones),batches

    def store_memory(self,state,action,action_prob,val,reward,done):
        self.states.append(state)
        self.actions.append(action)
        self.action_probs.append(action_prob)
        self.rewards.append(reward)
        self.vals.append(val)
        self.dones.append(done)

    def clear_memory(self):
        self.states = []
        self.actions= []
        self.action_probs = []
        self.rewards = []
        self.vals = []
        self.dones = []


In [63]:
############################ Actor Network ######################################

## initialize actor network and critic network


class ActorNwk(nn.Module):
    def __init__(self,input_dim,out_dim,
                 adam_lr,
                 chekpoint_file,
                 hidden1_dim=256,
                 hidden2_dim=256
                 ):
        super(ActorNwk, self).__init__()
    
        self.actor_nwk = nn.Sequential(
            nn.Linear(*input_dim,hidden1_dim),
            nn.ReLU(),
            nn.Linear(hidden1_dim,hidden2_dim),
            nn.ReLU(),
            nn.Linear(hidden2_dim,out_dim),
            nn.Softmax(dim=-1)
        )

        self.checkpoint_file = chekpoint_file
        self.optimizer = torch.optim.Adam(params=self.actor_nwk.parameters(),lr=adam_lr)

        self.device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
        self.to(self.device)


    def forward(self,state):
        out = self.actor_nwk(state)
        dist = Categorical(out)
        return dist


In [64]:
############################### Crirtic Network ######################################

class CriticNwk(nn.Module):
    def __init__(self,input_dim,
                 adam_lr,
                 chekpoint_file,
                 hidden1_dim=256,
                 hidden2_dim=256
                 ):
        super(CriticNwk, self).__init__()

        self.critic_nwk = nn.Sequential(
            nn.Linear(*input_dim,hidden1_dim),
            nn.ReLU(),
            nn.Linear(hidden1_dim,hidden2_dim),
            nn.ReLU(),
            nn.Linear(hidden2_dim,1),

        )

        self.checkpoint_file = chekpoint_file
        self.optimizer = torch.optim.Adam(params=self.critic_nwk.parameters(),lr=adam_lr)

        self.device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
        self.to(self.device)


    def forward(self,state):
        out = self.critic_nwk(state)
        return out

In [98]:
## Initilaize an Agent will will be able to train the model

############################# Agent ########################################3

## agent

class Agent():
    def __init__(self, gamma, policy_clip,lamda, adam_lr,
                 n_epochs, batch_size, state_dim, action_dim):

        self.gamma = gamma
        self.policy_clip = policy_clip
        self.lamda  = lamda
        self.n_epochs = n_epochs

        self.actor = ActorNwk(input_dim=state_dim,out_dim=action_dim,adam_lr=adam_lr,chekpoint_file='tmp/actor')
        self.critic = CriticNwk(input_dim=state_dim,adam_lr=adam_lr,chekpoint_file='tmp/ctitic')
        self.memory = PPOMemory(batch_size)

    def store_data(self,state,action,action_prob,val,reward,done):
        self.memory.store_memory(state,action,action_prob,val,reward,done)

    def choose_action(self, state):
        state = torch.tensor([state], dtype=torch.float).to(self.actor.device)

        dist = self.actor(state)
        ## sample the output action from a categorical distribution of predicted actions
        action = dist.sample()
        probs = torch.squeeze(dist.log_prob(action)).item()
        action = torch.squeeze(action).item()

        ## value from critic model
        value = self.critic(state)
        value = torch.squeeze(value).item()

        return action, probs, value

    def calculate_advanatage(self,reward_arr,value_arr,dones_arr):
        time_steps = len(reward_arr)
        advantage = np.zeros(len(reward_arr), dtype=np.float32)

        for t in range(0,time_steps-1):
            discount = 1
            running_advantage = 0
            for k in range(t,time_steps-1):
                if int(dones_arr[k]) == 1:
                    running_advantage += reward_arr[k] - value_arr[k]
                else:
                    running_advantage += reward_arr[k] + (self.gamma*value_arr[k+1]) - value_arr[k]

                running_advantage = discount * running_advantage
                discount *= self.gamma * self.lamda

            advantage[t] = running_advantage
        advantage = torch.tensor(advantage).to(self.actor.device)
        return advantage

    def learn(self):
        for _ in range(self.n_epochs):

            ## initially all will be empty arrays
            state_arr, action_arr, old_prob_arr, value_arr,\
            reward_arr, dones_arr, batches = \
                    self.memory.generate_batches()

            advantage_arr = self.calculate_advanatage(reward_arr,value_arr,dones_arr)
            values = torch.tensor(value_arr).to(self.actor.device)

            for batch in batches:
                states = torch.tensor(state_arr[batch], dtype=torch.float).to(self.actor.device)
                old_probs = torch.tensor(old_prob_arr[batch]).to(self.actor.device)
                actions = torch.tensor(action_arr[batch]).to(self.actor.device)

                dist = self.actor(states)
                critic_value = self.critic(states)

                critic_value = torch.squeeze(critic_value)

                new_probs = dist.log_prob(actions)
                prob_ratio = new_probs.exp() / old_probs.exp()
                #prob_ratio = (new_probs - old_probs).exp()
                weighted_probs = advantage_arr[batch] * prob_ratio
                weighted_clipped_probs = torch.clamp(prob_ratio, 1-self.policy_clip,
                        1+self.policy_clip)*advantage_arr[batch]
                actor_loss = -torch.min(weighted_probs, weighted_clipped_probs).mean()

                returns = advantage_arr[batch] + values[batch]
                critic_loss = (returns-critic_value)**2
                critic_loss = critic_loss.mean()

                total_loss = actor_loss + 0.5*critic_loss
                self.actor.optimizer.zero_grad()
                self.critic.optimizer.zero_grad()
                total_loss.backward()
                self.actor.optimizer.step()
                self.critic.optimizer.step()

        self.memory.clear_memory()

In [66]:
import gym
import numpy as np
import pandas as pd

LONG = 1
NOTHING = 0
SHORT = -1
class FutureTradingEnv(gym.Env):
    def __init__(self, data):
        super(FutureTradingEnv, self).__init__()
        self.data = data
        self.current_step = 0
        self.balance = 10000
        self.holdings = 0
        self.action_space = gym.spaces.Discrete(3)
        self.observation_space = gym.spaces.Box(low=-np.inf, high=np.inf, shape=(7,), dtype=np.float32)
        self.position = NOTHING
        self.avg_price = 0
        self.fee_rate = 0.02
        self.leverage = 1
    
    def reset(self):
        self.current_step = 0
        self.balance = 10000
        self.holdings = 0
        self.position = NOTHING
        self.avg_price = 0

        return self._next_observation()
    
    def _next_observation(self):
        return np.array([self.balance, self.holdings, self.data.iloc[self.current_step]['Open'],
                         self.data.iloc[self.current_step]['Close'], self.data.iloc[self.current_step]['CHG'],
                         self.data.iloc[self.current_step]['stocRSI'], self.data.iloc[self.current_step]['MACD']], dtype=np.float32)
    
    '''
    action 0 : LONG
    action 1 : SELL
    action 2 : SHORT
    '''
    def step(self, action):
        price = self.data.iloc[self.current_step]['Close']
        reward = 0
        self.current_step += 1
        done = self.current_step >= len(self.data) - 1
        truncated = False

        if action == 0 and self.position == NOTHING and self.balance > 0:
            self.avg_price = price
            self.holdings += (self.balance*self.leverage)/price
            self.balance = 0
            self.position = LONG
            
        elif action == 0 and self.position == SHORT and self.holdings > 0:
            #sell short
            self.balance += self.holdings*(2*self.avg_price-price)
            reward = self.holdings*(self.avg_price-price)
            self.holdings = 0
            #buy long
            self.avg_price = price
            self.holdings += (self.balance*self.leverage)/price
            self.balance = 0
            self.position = LONG
            
        elif action == 1 and self.position == LONG and self.holdings > 0:
            self.balance += self.holdings*price
            reward = self.holdings*(price-self.avg_price)
            self.holdings = 0
            self.avg_price = 0
            self.position = NOTHING
        
        elif action == 1 and self.position == SHORT and self.holdings > 0:
            self.balance += self.holdings*(2*self.avg_price-price)
            reward = self.holdings*(self.avg_price-price)
            self.holdings = 0
            self.avg_price = 0
            self.position = NOTHING
            
        elif action == 2 and self.position == NOTHING and self.balance > 0:
            self.avg_price = price
            self.holdings += (self.balance*self.leverage)/price
            self.balance = 0
            self.position = SHORT
            
        elif action == 2 and self.position == LONG and self.holdings > 0:
            #sell long
            self.balance += self.holdings*price
            reward = self.holdings*(price-self.avg_price)
            self.holdings = 0
            self.avg_price = 0
            #buy short
            self.avg_price = price
            self.holdings += (self.balance*self.leverage)/price
            self.balance = 0
            self.position = SHORT
            
            
        if done and self.position == SHORT and self.holdings > 0:
            reward = self.holdings*(self.avg_price-price)
            self.balance += self.holdings*price
            self.holdings = 0
        elif done and self.position == LONG and self.holdings > 0:
            reward = self.holdings*(price-self.avg_price)
            self.balance += self.holdings*price
            self.holdings = 0

        if self.balance < 8000 and self.holdings == 0:
            print('truncated')
            truncated = True
        #print(f'action: {action}, balance: {self.balance}, holdings: {self.holdings}')
        return self._next_observation(), reward, done, truncated

In [94]:
class SpotTradingEnv(gym.Env):
    def __init__(self, data):
        super(SpotTradingEnv, self).__init__()
        self.data = data
        self.current_step = 0
        self.balance = 10000
        self.holdings = 0
        self.action_space = gym.spaces.Discrete(3)
        self.observation_space = gym.spaces.Box(low=-np.inf, high=np.inf, shape=(7,), dtype=np.float32)
        self.avg_price = 0

    def reset(self):
        self.current_step = 0
        self.balance = 10000
        self.holdings = 0

        return self._next_observation()
    
    def _next_observation(self):
        return np.array([self.balance, self.holdings, self.data.iloc[self.current_step]['Open'],
                         self.data.iloc[self.current_step]['Close'], self.data.iloc[self.current_step]['CHG'],
                         self.data.iloc[self.current_step]['stocRSI'], self.data.iloc[self.current_step]['MACD']], dtype=np.float32)
    
    '''
    action 0 : 매수
    action 1 : 관망
    action 2 : 매수

    마지막 step에선 현재 보유 중인 코인을 전부 매도
    '''
    def step(self, action):
        price = self.data.iloc[self.current_step]['Close']
        reward = -10000
        self.current_step += 1
        done = self.current_step >= len(self.data) - 1
        truncated = False
        
        if action == 0 and self.balance > 0:
            self.avg_price = price
            self.holdings += self.balance/price
            self.balance = 0
        elif action == 2 and self.holdings > 0:
            self.balance += self.holdings*price
            reward = self.holdings*price - self.holdings*self.avg_price
            self.holdings = 0
            self.avg_price = 0
        
        if done and self.holdings > 0:
            reward = self.holdings*price - self.avg_price
            self.balance += self.holdings*price
            self.holdings = 0
        
        if self.balance < 8000 and self.holdings == 0:
            print('truncated')
            truncated = True

        return self._next_observation(), reward, done, truncated

In [99]:
### Train the model


import numpy as np
import matplotlib.pyplot as plt

data = pd.read_csv(f'/workspace/data/preprocess/BTCUSDT/BTCUSDT-1d.csv', index_col=0)
data = data[['Open','Close','Volume','CHG','stocRSI','MACD']]

#env = FutureTradingEnv(data)
env = SpotTradingEnv(data)
N = 20
batch_size = 5
n_epochs = 4
alpha = 0.0003
agent = Agent(state_dim=env.observation_space.shape,
              action_dim=env.action_space.n,
              batch_size=batch_size,
              n_epochs=n_epochs,
              policy_clip=0.2,
              gamma=0.99,lamda=0.95,
              adam_lr=alpha)
n_games = 300
score_history = []
learn_iters = 0
avg_score = 0
n_steps = 0
for i in range(n_games):
    current_state = env.reset()
    terminated,truncated = False,False
    done = False
    score = 0
    rewards = []
    while not done:
        action, prob, val = agent.choose_action(current_state)
        next_state, reward, terminated, truncated = env.step(action)
        done = 1 if (terminated or truncated) else 0
        n_steps += 1
        score += reward
        agent.store_data(current_state, action, prob, val, reward, done)
        if n_steps % N == 0:
            agent.learn()
            learn_iters += 1
        current_state = next_state
        rewards.append(reward)
        
    score_history.append(score)
    avg_score = np.mean(score_history[-100:])
    print(f'episode: {i}, balance: {env.balance}, holdings: {env.holdings}, reward: {sum(rewards)}, time_steps: {n_steps}')
    #print('episode', i, 'score %.1f' % env.balance, 'avg score %.1f' % avg_score,
    #        'time_steps', n_steps, 'learning_steps', learn_iters)

    if i % 10 == 0:
        total_reward = sum(rewards)
        print(f'Episode {i}, Total Reward: {total_reward}')



episode: 0, balance: 10000, holdings: 0, reward: -23270000, time_steps: 2327
Episode 0, Total Reward: -23270000
episode: 1, balance: 10000, holdings: 0, reward: -23270000, time_steps: 4654


KeyboardInterrupt: 